# Request boundaries — testing the decision logic (Q29)

The participant round records turns, not requests, so a rule has to decide
where one request ends and the next begins. This notebook tests that rule
against a simulated run, where the true boundaries are known: **one execution
is one request**.

**Status: run on the D82 probe (`287e7780`), pooled with `42cab17a`.** The run it exists for
includes the two multi-step seeds (D81). Those are the only cases with a
request that changes shape mid-way, and so the only source of the split
error. Before review, the author asked for a probe at n = 3 to decide whether they
stay (D82). This notebook reads that probe together with the 29-case run
`42cab17a`, which supplies the merge-error side.

## The rule under test (`shapes.segment_with`)

For each turn after the first in a Tonic run, the first rule that applies
decides:

| # | signal | reads | decision |
|---|---|---|---|
| 1 | **reference** | the turn's tool arguments against the ids the current request's earlier turns offered or touched (reply place refs, tool-argument ids) | acts on one → **same request** |
| 2 | **slash** | `study_suggestions.agent_intent` | carries a command → **new request** |
| 3 | **shape** | the turn's shape (`shapes7`) | shape changed → new request; no tool → same request |

Reference and slash are **optional evidence**. The agent reaches every path
without a command, so when a slash is absent the rule falls back to the shape.
Reference outranks slash because a member can use a command to continue
("/plan put the second one on Saturday"). The rule is **causal**: a turn is
decided from itself and earlier turns only. The same function therefore
labels a run turn by turn as it happens, or in one pass afterwards, with
identical results.

## What is measured

- **Split** (one request cut in two): boundaries drawn *inside* an execution.
- **Merge** (two requests read as one): executions fused into synthetic runs,
  and the seams a rule misses.
- **Each signal's share:** four variants, from shape only to both signals.
- **Slashes stripped:** the same turns with every command removed, standing
  in for a member who types free text. This is an approximation: the agent
  acted under the command's toolset, and stripping removes the signal, not
  its effect on the agent.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parents[1]))

import pandas as pd

from src import survey, shapes
from src import segmentation_bench as bench
from src.cases import load_cases

pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)

# The runs under test: the D82 probe (the two multi-step seeds, n = 3) and the
# 29-case run 42cab17a, pooled so fused runs mix the new executions with the
# old ones. Keys carry the run, the case and the repetition.
RUN_IDS = ["287e7780-2409-450e-a5d1-234b43146390", "42cab17a-4903-4e39-b00c-ace76b7929e4"]
MULTI_STEP = ["scheduling_dinner_from_offer", "scheduling_afternoon_then_tram"]

executions, spanning = {}, []
for rid in RUN_IDS:
    print(survey.headline(rid))
    print()
    loaded = survey.turn_signals(rid)
    spanning += loaded.pop(None)
    executions.update({f"{rid[:8]}:{k}": v for k, v in loaded.items()})
present = sorted(k for k in executions if k.split(":")[1].split("#")[0] in MULTI_STEP)
print(f"{len(executions)} executions loaded; spanning more than one Tonic run: {spanning or 'none'}")
print(f"multi-step executions: {len(present)}")
assert not spanning, "an execution is not one request here; the ground truth does not hold"


run 287e7780  started 2026-09-24T07:27:36
  notes         PROBE multi-step seeds n=3 (D82) - NOT a baseline; unreviewed cases
  agent         gemini-3.5-flash (released 2026-05-19)
  api           0.1.3 @ 5af96abf452b
  suite         71d11f9749ba
  executions    6  (+0 superseded)
  failed        0
  scored        6, mean 8.17
  cost          $0.3467



run 42cab17a  started 2026-09-23T17:52:07
  notes         FULL SUITE n=1 post-D74..D80 - verification
  agent         gemini-3.5-flash (released 2026-05-19)
  api           0.1.3 @ 5af96abf452b
  suite         031589fd9658
  executions    29  (+0 superseded)
  failed        0
  scored        29, mean 9.34
  cost          $0.6498



35 executions loaded; spanning more than one Tonic run: none
multi-step executions: 6


## 1. Split error — within executions

Every execution is one request, so every value above 1 is a split. Only
multi-step executions can split. For the others the rule's first boundary is
never inside them.

In [2]:
within = pd.DataFrame({v: bench.within(executions, v) for v in bench.VARIANTS})
within_stripped = pd.DataFrame({v: bench.within(bench.strip_slashes(executions), v) for v in bench.VARIANTS})

split = pd.DataFrame({
    "as recorded": (within > 1).sum(),
    "slashes stripped": (within_stripped > 1).sum(),
}).T
print(f"executions cut into more than one request, of {len(within)}:")
split

executions cut into more than one request, of 35:


,shape only,+ slash,+ reference,+ slash + reference
as recorded,6,6,0,0
slashes stripped,6,6,0,0


In [3]:
# The multi-step executions turn by turn: what each signal saw and decided.
rows = []
for case in present:
    for i, t in enumerate(executions[case], start=1):
        rows.append({"case": case, "turn": i, "shape": t.label, "slash": t.intent or "-",
                     "acts on": len(t.acted_on), "offers": len(t.offered)})
    rows.append({"case": case, "turn": "requests", **{v: within.at[case, v] for v in bench.VARIANTS}})
pd.DataFrame(rows).fillna("") if rows else "no multi-step executions in this run"


,case,turn,shape,slash,acts on,offers,shape only,+ slash,+ reference,+ slash + reference
0,287e7780:scheduling_afternoon_then_tram#1,1,scheduling.draft,plan,3.0,5.0,,,,
1,287e7780:scheduling_afternoon_then_tram#1,2,other,-,6.0,8.0,,,,
2,287e7780:scheduling_afternoon_then_tram#1,requests,,,,,2.0,2.0,1.0,1.0
3,287e7780:scheduling_afternoon_then_tram#2,1,scheduling.draft,plan,3.0,3.0,,,,
4,287e7780:scheduling_afternoon_then_tram#2,2,other,-,5.0,5.0,,,,
5,287e7780:scheduling_afternoon_then_tram#2,requests,,,,,2.0,2.0,1.0,1.0
6,287e7780:scheduling_afternoon_then_tram#3,1,scheduling.draft,plan,3.0,3.0,,,,
7,287e7780:scheduling_afternoon_then_tram#3,2,other,-,4.0,4.0,,,,
8,287e7780:scheduling_afternoon_then_tram#3,requests,,,,,2.0,2.0,1.0,1.0
9,287e7780:scheduling_dinner_from_offer#1,1,venue.search,find,0.0,5.0,,,,


**Predictions, stated before the run.**

- `scheduling_dinner_from_offer`: shape alone cuts it in two (venue.search,
  then scheduling.draft). **With reference it is one**, because the draft
  acts on a place the search's reply offered.
- `scheduling_afternoon_then_tram`: cut in two **under every variant**. The
  move acts on the tram, which the draft did not offer: the continuity is in
  the member's words ("the end of that afternoon"), not in any id. If that
  holds, draft → edit is a split no signal in the record can repair, and it is
  reported as a known limit rather than tuned away.

If either prediction fails, the rule or the ids it reads are not what this
notebook assumes, and that needs finding before Q29 is ruled on.

**Result on the probe.**

- **Dinner: as predicted.** Shape alone cuts every repetition in two;
  reference joins all three, because each draft holds a place the search's
  reply offered. D82's keep rule, every repetition, is met on this side.
- **Afternoon: the prediction failed**, and the reason matters more than the
  miss. In all three repetitions the agent's second turn called
  `propose_plan` *and* `move_slot` (shape `other`): it re-proposed the draft,
  reusing its own places, and moved the tram in the same turn. Reference
  joins the two turns through the **re-proposed draft's place ids**, not
  through the tram. So the link is real in the record but incidental. An
  agent that moved the tram without re-drafting would have been split, as
  predicted, and the same holds for a participant's follow-up whose only link
  is its wording.
- **Slash does not change either seed.** Both open with a command and
  continue without one, so a slash never marks the second turn.

## 2. Merge error — fused runs

All ordered pairs, then random runs of 3, 5 and 8 requests (2 000 each, seed
28), each scored under every variant, as recorded and with slashes stripped.
`false boundaries per run` is the split error again, now inside fused runs.

In [4]:
samples = {"all pairs": bench.all_pairs(executions)}
for k in (3, 5, 8):
    samples[f"{k} requests"] = bench.random_runs(executions, k, 2000)

rows = []
for label, source in (("as recorded", executions), ("slashes stripped", bench.strip_slashes(executions))):
    for sample, sequences in samples.items():
        for variant in bench.VARIANTS:
            rows.append({"slashes": label, "sample": sample, "variant": variant,
                         **bench.score(source, variant, sequences)})
fused = pd.DataFrame(rows)
table = fused.pivot_table(index=["slashes", "sample"], columns="variant",
                          values="boundaries found", sort=False)[list(bench.VARIANTS)]
print("boundaries found (1 - merge rate)")
table.map("{:.1%}".format)

boundaries found (1 - merge rate)


variant                     shape only + slash + reference + slash + reference
slashes          sample                                                       
as recorded      all pairs       84.7%   93.8%       84.2%               92.8%
                 3 requests      82.8%   92.4%       82.0%               90.4%
                 5 requests      82.9%   92.7%       82.0%               90.8%
                 8 requests      83.0%   92.9%       82.2%               91.1%
slashes stripped all pairs       84.7%   84.7%       84.2%               84.2%
                 3 requests      82.8%   82.8%       82.0%               82.0%
                 5 requests      82.9%   82.9%       82.0%               82.0%
                 8 requests      83.0%   83.0%       82.2%               82.2%

In [5]:
print("false boundaries per run (split error inside fused runs)")
fused.pivot_table(index=["slashes", "sample"], columns="variant",
                  values="false boundaries per run", sort=False)[list(bench.VARIANTS)].map("{:.3f}".format)

false boundaries per run (split error inside fused runs)


variant                     shape only + slash + reference + slash + reference
slashes          sample                                                       
as recorded      all pairs       0.343   0.343       0.000               0.000
                 3 requests      0.508   0.508       0.000               0.000
                 5 requests      0.861   0.861       0.000               0.000
                 8 requests      1.395   1.395       0.000               0.000
slashes stripped all pairs       0.343   0.343       0.000               0.000
                 3 requests      0.508   0.508       0.000               0.000
                 5 requests      0.861   0.861       0.000               0.000
                 8 requests      1.395   1.395       0.000               0.000

## 3. What reference merges that it should not

Reference joins a turn to the request before it when the turn acts on an id
that request offered. Two *different* requests can share a place: two Belém
mornings both drafting the same bakery, for example. Fused, they read as one.
These are the pairs where adding reference loses a seam that shape and slash
alone would have found.

In [6]:
wrong = []
for a, b in bench.all_pairs(executions):
    base = bench.score(executions, "+ slash", [(a, b)])["runs exact"]
    ref = bench.score(executions, "+ slash + reference", [(a, b)])["runs exact"]
    if base == 1 and ref == 0:
        wrong.append({"first": a, "second": b})
print(f"pairs reference merges wrongly: {len(wrong)} of {len(bench.all_pairs(executions))}")
pd.DataFrame(wrong).head(20) if wrong else "none"


pairs reference merges wrongly: 6 of 1190


,first,second
0,42cab17a:scheduling_belem_morning#1,42cab17a:scheduling_belem_morning__dir2_on_the...
1,42cab17a:scheduling_belem_morning#1,42cab17a:scheduling_belem_morning__dir3_a_full...
2,42cab17a:scheduling_belem_morning__dir2_on_the...,42cab17a:scheduling_belem_morning#1
3,42cab17a:scheduling_belem_morning__dir2_on_the...,42cab17a:scheduling_belem_morning__dir3_a_full...
4,42cab17a:scheduling_belem_morning__dir3_a_full...,42cab17a:scheduling_belem_morning#1
5,42cab17a:scheduling_belem_morning__dir3_a_full...,42cab17a:scheduling_belem_morning__dir2_on_the...


## 4. Reading this run

*The D82 probe (`287e7780`, 2 seeds × 3), pooled with `42cab17a` (29 cases):
35 executions, 6 of them multi-step. Every execution is exactly one Tonic run.*

**Split error: reference removes it, and nothing else does.** Without
reference, all 6 multi-step executions are cut in two, which in fused runs
means 0.34 false boundaries per pair and 1.4 per run of eight requests. With
reference, both numbers are 0. Slash helps with neither, because both seeds
continue without a command.

**D82's keep rule is met for the dinner seed.** Reference joins search →
draft in 3 of 3 repetitions, through the offered place's id. Its false merges
across all 1 190 fused pairs are the same 6 as before, all between the three
Belém drafts, which share places. None involves the new executions.

**The afternoon seed was joined too, but not the way it was meant to be.** In
all three repetitions turn two re-proposed the draft and moved the tram in
one turn. The link is the re-proposed draft's own place ids, so it depends on
the agent choosing to re-draft. A follow-up whose only link is its wording
("the end of that afternoon") would still be split.

**Merge error with the new executions pooled:** slash raises the boundaries
found from 84.7 % to 93.8 % over all pairs. Reference costs about one point
(92.8 %). With slashes stripped, reference is the only signal left, and it
trades that point for removing every split.

**What this means for the study.** Reference reliably marks one kind of
follow-up: acting on something the agent just offered. It does not mark a
follow-up linked only by wording. Slash only helps where members type
commands. Neither threshold nor rule is fixed here; that is Q29.